In [ ]:
acute_df = df[
    df['Disease'].str.strip().str.lower()
    == 'acute diarrhoeal disease'
].copy()

acute_df['Date'] = pd.to_datetime(
    acute_df[['year', 'mon', 'day']].rename(columns={'mon': 'month'}),
    errors='coerce'
)

# Convert date to Monday of its week
acute_df['Week'] = (
    acute_df['Date']
    - pd.to_timedelta(acute_df['Date'].dt.weekday, unit='D')
)

weekly_state = (
    acute_df
    .groupby(['state_ut', 'Week'])
    .agg(
        total_cases=('Cases', 'sum'),
        total_deaths=('Deaths', 'sum'),
        reporting_districts=('district', 'nunique')
    )
    .reset_index()
)

state_week_counts = (
    weekly_state
    .groupby('state_ut')
    .agg(
        unique_weeks=('Week', 'nunique'),
        first_week=('Week', 'min'),
        last_week=('Week', 'max'),
        total_cases=('total_cases', 'sum')
    )
    .reset_index()
    .sort_values('unique_weeks', ascending=False)
)

print(state_week_counts.to_string(index=False))